In [1]:
import sys
sys.path.insert(0,'..')
from warnings import filterwarnings
filterwarnings("ignore")
%load_ext autoreload
%autosave 180

Autosaving every 180 seconds


In [2]:
%autoreload
import os
import random
import torch
import torch.nn as nn
import pandas as pd
import numpy as np
from torch.optim import Adam
from torch.utils.data import DataLoader
from source.version0.data import trainLoader
from source.version0.model import EfficientModel
from source.version0.train import trainModel
from source.version0.loss import CELoss
from torch.optim.lr_scheduler import CosineAnnealingWarmRestarts as CosLR

In [3]:
def seed(seed):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = True
    return None

seed(2017)

In [4]:
def train(fold):
    loader = {}
    loader['image_path'] = '../../data/raw/train_images/'
    loader['label_path'] = '../../data/raw/data.csv'
    loader['fold_idx'] = fold
    train, valid = trainLoader(**loader)
    params = {}
    params['batch_size'] = 7
    params['num_workers'] = 3
    params['drop_last'] = True
    train = DataLoader(train, **params, shuffle=True)
    valid = DataLoader(valid, **params, shuffle=False)
    model = EfficientModel()
    model = model.to('cuda:0')
    optimizer = Adam(model.parameters(), lr=1e-04, weight_decay=1e-6)
    schedular = CosLR(optimizer, T_0=10, T_mult=1, eta_min=1e-6, last_epoch=-1)
    trainer = {}
    trainer['model'] = model
    trainer['train_data'] = train
    trainer['valid_data'] = valid
    trainer['loss_fn'] = nn.CrossEntropyLoss()
    trainer['optimizer'] = optimizer
    trainer['save_path'] = '../../model/version0/model_{}.pt'.format(fold)
    trainer['epochs'] = 10
    trainer['batch'] = 7
    trainer['scheduler'] = schedular
    trainModel(**trainer)
    model.cpu()
    del model
    return None

In [ ]:
train(0)

Train Images: 17117 Valid Images: 4280


100% 17115/17115 [13:54<00:00, 20.52it/s, trn_ls=0.6272, val_ls=0.4274, val_mt=0.8562]
100% 17115/17115 [13:27<00:00, 21.19it/s, trn_ls=0.4520, val_ls=0.3920, val_mt=0.8667]
100% 17115/17115 [13:27<00:00, 21.18it/s, trn_ls=0.4135, val_ls=0.3566, val_mt=0.8787]
100% 17115/17115 [13:27<00:00, 21.19it/s, trn_ls=0.3776, val_ls=0.3744, val_mt=0.8782]
100% 17115/17115 [13:27<00:00, 21.18it/s, trn_ls=0.3623, val_ls=0.3588, val_mt=0.8815]
100% 17115/17115 [13:28<00:00, 21.18it/s, trn_ls=0.3424, val_ls=0.3602, val_mt=0.8810]
100% 17115/17115 [13:28<00:00, 21.16it/s, trn_ls=0.3236, val_ls=0.3483, val_mt=0.8805]
100% 17115/17115 [13:27<00:00, 21.18it/s, trn_ls=0.3071, val_ls=0.3455, val_mt=0.8826]
 21% 3605/17115 [02:41<09:42, 23.19it/s, trn_ls=0.30320]

In [ ]:
train(1)

In [ ]:
train(2)

In [ ]:
train(3)

In [ ]:
train(4)